In [1]:
# import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.ensemble import HistGradientBoostingClassifier

# load data

In [ ]:
# load dataset
if not os.path.exists("bank-additional-full.csv"):
    import urllib.request
    url = "https://raw.githubusercontent.com/Graceagaba/Data_mining_assignment_1/main/bank-additional-full.csv"
    urllib.request.urlretrieve(url, "bank-additional-full.csv")
df = pd.read_csv("bank-additional-full.csv", sep=";")
print(df.shape)
df.head()

(41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [4]:
# any missing values
df.isnull().sum()

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

there are no missing values

In [5]:
# any duplicates
df.duplicated().sum()

np.int64(12)

There are 12 duplicate rows

In [6]:
# count of y values
print(df['y'].value_counts())

y
no     36548
yes     4640
Name: count, dtype: int64


36548 customers (88.7%) said no and 4640(11.3%)customers said yes.
There is a class imbalance in the dataset which might be a problem for the model.This means a mod; can score about 89% accuracy by predicting no for every customer while finding noine for those who actually subscribe


In [7]:
# number of 'unknown' values
categorical_cols = df.select_dtypes(include="object").columns
for c in categorical_cols:
    print(c, (df[c] == "unknown").sum())

job 330
marital 80
education 1731
default 8597
housing 990
loan 990
contact 0
month 0
day_of_week 0
poutcome 0
y 0


In [8]:
# pdays column
print(df["pdays"].value_counts().head())
print((df["pdays"] == 999).mean().round(4))

pdays
999    39673
3        439
6        412
4        118
9         64
Name: count, dtype: int64
0.9632


In [9]:
# duration column
print(df.groupby("y")["duration"].mean().round(1))

y
no     220.8
yes    553.2
Name: duration, dtype: float64


## data preparation

In [10]:
# handle duplicates
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)

(41176, 21)


there were 12 exact duplicate rows ,so thhey were dropped so that they dont appear in the train and test sets 

In [11]:
# handle duration
# The call length is known after the call ends so it cannot help decide who to call.keeping it leads to data leakage
df = df.drop(columns=['duration'])
print(df.shape)


(41176, 20)


In [12]:
# handle pdays
df["previously_contacted"] = (df["pdays"] != 999).astype(int)
df = df.drop(columns=["pdays"])
print(df["previously_contacted"].value_counts())


previously_contacted
0    39661
1     1515
Name: count, dtype: int64


About 96.3% of the rows in pdays is 999 meaning they were never contacted before bot a number of days.so they are replaced with 0/1 flag,previously_contacted so that the model is not misled by a false number.

In [13]:
# convert target to numbers
df['y'] = (df['y'] == 'yes').astype(int)
print(df['y'].mean().round(4))

0.1127


The yes rate was 11.27% and y was coverted as 1 for yes and 0 for no sice models need numbers.

# Short Report
The dataset contains 41,188 calls and 21 variables, with no missing values and 12 duplicate rows. 
The target is imbalanced since 36548 customers (88.7%) declined and 4640(11.3%) subscribed. 
The goal is to predict from the information known before a call whether a customer subscribes to a term deposit, so the bank can choose who to call.

A model that always predicts “no” would therefore achieve 88.7% accuracy without identifying a single subscriber, so accuracy is not a suitable metric and so precision, recall and AUC- based measures will be used instead.

The data has 12 duplicate rows and 6 columns that use the word ‘unknown’ as a hidden form of missing data. The duplicate rows were removed so that they don’t appear in both train and test datasets. Deleting the unknown values removes a big percentage of the data so they were be kept as their own category.

In the pdays column, 96.3 % of rows pdays equals 999, a code for never contacted before in an earlier campaign according to this dataset. This means reading 999 as a number would tell a model the customer was last called 999 days ago which is not true. pdays were replaced with 0/1 flag as previously-contacted.

As for duration; customers who subscribed spoke for 553 seconds on average while those who declined spoke for 221 seconds. The call length can only be known once the call has ended, therefore this information cannot be used to determine whom to call so it will also be dropped to avoid data leakage.


# split dataset 

In [14]:
X = df.drop(columns='y')
y = df['y']
print(X.shape, y.shape)

(41176, 19) (41176,)


In [15]:
# split the data
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(4), y_test.mean().round(4))


(32940, 19) (8236, 19)
0.1127 0.1127


In [16]:
# majority- class baseline
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

print("Accuracy :", round(accuracy_score(y_test, pred), 4))
print("Recall   :", recall_score(y_test, pred))
print("Precision:", precision_score(y_test, pred, zero_division=0))

Accuracy : 0.8873
Recall   : 0.0
Precision: 0.0


The DummyClassifier model ignores all other clues and predicts what is most common in training and so there are no 'yes' predictions as well as 0 precision because of the zero-division. This means the bank would make no calls and earn no deposits.

A majority – class baseline that predicts “no” for every customer achieves 88.73% accuracy on the test set but its recall is 0. This means it doesn’t identify any of the 928 customers in the test set who would subscribe and its precision is also undefined because it didn’t predict ‘yes’. This therefore means that accuracy is misleading on an imbalanced data and so the models will be judged on recall, precision, F1, ROC-AUC and PR-AUC against this baseline.

The split was stratified on the target so that the training and test sets both keep the 11.27% subscription rate.

## task 2 :  Comparing the 3 models

In [17]:
categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()
numerical_cols = X_train.select_dtypes(include="number").columns.tolist()
print(len(categorical_cols), categorical_cols)
print(len(numerical_cols), numerical_cols)

10 ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']
9 ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'previously_contacted']


In [18]:
# preprocessor
preprocess = ColumnTransformer([
    ("text", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("numbers", StandardScaler(), numerical_cols)
])

In [19]:
# logistic regresion with cross-validation
logreg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(logreg, X_train, y_train, cv=cv,
                        scoring=["roc_auc", "average_precision"])

print("ROC-AUC :", scores["test_roc_auc"].mean().round(4))
print("PR-AUC  :", scores["test_average_precision"].mean().round(4))

ROC-AUC : 0.7905
PR-AUC  : 0.4467


In [20]:
# Random forest

rf = Pipeline([
    ("prep", preprocess),
    ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                     class_weight="balanced_subsample",
                                     n_jobs=-1, random_state=42))
])

scores_rf = cross_validate(rf, X_train, y_train, cv=cv,
                           scoring=["roc_auc", "average_precision"])

print("ROC-AUC :", scores_rf["test_roc_auc"].mean().round(4))
print("PR-AUC  :", scores_rf["test_average_precision"].mean().round(4))

ROC-AUC : 0.795
PR-AUC  : 0.4655


In [21]:
# Gradient boosting

gb = Pipeline([
    ("prep", preprocess),
    ("model", HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300,
                                             max_leaf_nodes=15,
                                             l2_regularization=1.0,
                                             random_state=42))
])

scores_gb = cross_validate(gb, X_train, y_train, cv=cv,
                           scoring=["roc_auc", "average_precision"])

print("ROC-AUC :", scores_gb["test_roc_auc"].mean().round(4))
print("PR-AUC  :", scores_gb["test_average_precision"].mean().round(4))

ROC-AUC : 0.8006
PR-AUC  : 0.4653


In [22]:
# compare the models
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic regression", "Random forest", "Gradient boosting"],
    "CV ROC-AUC": [scores["test_roc_auc"].mean(),
                   scores_rf["test_roc_auc"].mean(),
                   scores_gb["test_roc_auc"].mean()],
    "CV PR-AUC": [scores["test_average_precision"].mean(),
                  scores_rf["test_average_precision"].mean(),
                  scores_gb["test_average_precision"].mean()],
    "ROC-AUC spread": [scores["test_roc_auc"].std(),
                       scores_rf["test_roc_auc"].std(),
                       scores_gb["test_roc_auc"].std()],
}).round(4)
results

,Model,CV ROC-AUC,CV PR-AUC,ROC-AUC spread
0,Logistic regression,0.7905,0.4467,0.0075
1,Random forest,0.7950,0.4655,0.0088
2,Gradient boosting,0.8006,0.4653,0.0079


Gradient boosting is the highest ROC_AUC of 0.8006 and a PR_AUC of 0.4653 is almost level with that for  Random forest.
Logistic regression has the lowest for both metrics.    

In [23]:
# fit models on the whole train and test data

models = [("Logistic regression", logreg),
          ("Random forest", rf),
          ("Gradient boosting", gb)]

test_probs = {}
for name, model in models:
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    test_probs[name] = p
    print(name,
          "| ROC-AUC:", round(roc_auc_score(y_test, p), 4),
          "| PR-AUC:", round(average_precision_score(y_test, p), 4))

Logistic regression | ROC-AUC: 0.8002 | PR-AUC: 0.4465
Random forest | ROC-AUC: 0.8048 | PR-AUC: 0.484
Gradient boosting | ROC-AUC: 0.8146 | PR-AUC: 0.4945


The 3 approaches were compared using 5 fold stratified cross -validation on the training set and a single evaluation on te held-out test set.

Gradient boosting with ROC-AUC = 0.815,PR-AUC=0.495 was ranked first,followed by random forest with ROC-AUC=0.805,PR-AUC=0.484, then followed by logistic regression with ROC-AUC=0.800,PR-AUC=0.447.

All the 3 models outperfomed the majority-class baseline of PR-AUC=0.113.

The gap between gradient boosting and random forest in cross-validation was also smaller than the fold to fold standard deviation so the ranking of the top 2 models relies partially on the test-set result.Logistic regression model is the simplest and most interpretable but the weakest predictor.
A majority – class baseline that predicts “no” for every customer achieves 88.73% accuracy on the test set but its recall is 0. This means it doesn’t identify any of the 928 customers in the test set who would subscribe and its precision is also undefined because it didn’t predict ‘yes’. This therefore means that accuracy is misleading on an imbalanced data and so the models will be judged on recall, precision, F1, ROC-AUC and PR-AUC against this baseline.
The split was stratified on the target so that the training and test sets both keep the 11.27% subscription rate.


# Small Report
Three approaches were selected to cover a range of complexity.

Logistic regression is a simple, fast and interpretable benchmark. Random forest captures non-linear patterns and interactions between variables and needs little tuning. Gradient boosting builds trees sequentially, each correcting the errors of the last, and often performs best on tabular data like this.

The models were compared using 5-fold stratified cross-validation on the training set, so that model selection did not use the test set, and each was then evaluated once on the held-out test set. ROC-AUC and PR-AUC were used because they do not depend on a threshold, and PR-AUC is especially suited to a rare positive class. Class weights were used to handle the imbalance in logistic regression and random forest.

Gradient boosting is ranked first because it has the highest ROC-AUC in cross-validation and on the test set, and the highest test PR-AUC.
Random forest is second.Its cross-validation scores were almost identical to gradient boosting’s, with the ROC-AUC gap (0.006) smaller than the fold-to-fold standard deviation (about 0.008), so the test set decided the order.

Logistic regression is third, with the lowest scores on every measure, although it is the easiest to interpret.

All three models clearly beat the baseline: the best test PR-AUC (0.4945) is about 4.4 times the baseline’s 0.113. The top two models are close, so their ranking is supported by the test result rather than being conclusive.

# task 3 : choosing the decision threshold

In [24]:
oof = cross_val_predict(gb, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
print(len(oof))

32940


In [25]:
grid = np.arange(0.05, 0.80, 0.01)
f1s = [f1_score(y_train, (oof >= t).astype(int)) for t in grid]
best_f1_t = grid[int(np.argmax(f1s))]

print("Best F1 Threshold:", round(best_f1_t, 2), "| Max F1 Score:", round(max(f1s), 3))

Best F1 Threshold: 0.2 | Max F1 Score: 0.499


In [26]:
# final test
t_final = 0.20
gb.fit(X_train, y_train)
p_test = gb.predict_proba(X_test)[:, 1]
pred_test = (p_test >= t_final).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()

print("Customers called :", tp + fp, "of", len(y_test))
print("Subscribers found:", tp, "| missed:", fn)
print("Wasted calls     :", fp)
print("Precision        :", round(tp / (tp + fp), 3))
print("Recall           :", round(tp / (tp + fn), 3))

Customers called : 1187 of 8236
Subscribers found: 558 | missed: 370
Wasted calls     : 629
Precision        : 0.47
Recall           : 0.601


# the recommended threshold is 0.20.
The recommended threshold is 0.20. The bank should call every customer whose predicted probability of subscribing is 0.20 or higher, using gradient boosting.

The threshold was chosen on out-of-fold training predictions, where F1 peaks at about 0.20(F1 of about 0.50) and was then checked once on the test set. 
This selects 1,187 of 8,236 customers (14%). Of these, 558 subscribe (precision 47%, against 11% by random calling), reaching about 60% of all subscribers (recall).

The trade-off. At 0.20 there are 629 unnecessary calls (customers who decline) and 370 missed subscribers. Lowering the threshold finds more subscribers but wastes more calls: at 0.10, recall rises to 68% but precision falls to 36%. Raising it saves calls but misses more subscribers: at 0.30, recall falls to 50%. The default of 0.5 would find only 24% of subscribers.

The best threshold depends on the bank’s real costs. If a call costs 1 unit and a subscription earn 10, the profit-maximising threshold is about 0.10, because each missed subscriber is worth ten calls.
If calls are relatively costly, the threshold should be higher. 0.20 is a balanced choice until the bank supplies its actual figures.
